# Python and CLI for Data Science - Session 14

- *Course*: Python and CLI for Data Science
- *Session*: 14
- *Unit*: Scikit-Learn - Feature Engineering

### (Re)sources:
- [Python Data Science Handbook](https://jakevdp.github.io/PythonDataScienceHandbook/index.html) _by Jake VanderPlas (Code released under the MIT License)_


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import io

# Feature Engineering

- Real-world data often does come in a clean `[n_samples, n_features]` feature matrix
- An important step in machine learning *feature engineering*: taking information you have about your problem and turning it into numbers


- We will cover a few common examples of feature engineering tasks

## Categorical Features

- One common type of nonnumerical data is *categorical* data
- For example, imagine data on housing prices, and along with numerical features like "price" and "rooms," you also have "neighborhood" information

In [ ]:
data = [
    {"price": 850000, "rooms": 4, "neighborhood": "Queen Anne"},
    {"price": 700000, "rooms": 3, "neighborhood": "Fremont"},
    {"price": 650000, "rooms": 3, "neighborhood": "Wallingford"},
    {"price": 600000, "rooms": 2, "neighborhood": "Fremont"},
]

- You might be tempted to encode this data with a straightforward numerical mapping

In [ ]:
{"Queen Anne": 1, "Fremont": 2, "Wallingford": 3}

- This is generally not a useful approach in Scikit-Learn 
- Such a mapping would imply, for example, that *Queen Anne < Fremont < Wallingford*, or even that *Wallingford–Queen Anne = Fremont*


- One proven technique is to use *one-hot encoding*
- It effectively creates extra columns indicating the presence or absence of a category with a value of 1 or 0, respectively.
- When your data takes the form of a list of dictionaries, Scikit-Learn's ``DictVectorizer`` will do this for you

In [ ]:
from sklearn.feature_extraction import DictVectorizer

vec = DictVectorizer(sparse=False, dtype=int)
vec.fit_transform(data)

- The `neighborhood` column has been expanded into three separate columns
- To see the meaning of each column, you can inspect the feature names

In [ ]:
vec.get_feature_names_out()

- One-hot-encoding has one clear disadvantage: if your category has many possible values, this can *greatly* increase the size of your dataset
- However, because the encoded data contains mostly zeros, a sparse output can be a very efficient solution

In [ ]:
vec = DictVectorizer(sparse=True, dtype=int)
vec.fit_transform(data)

## Text Features

- Another common need is to featurize text
- One of the simplest methods of encoding text is by *word counts*
    - Take each snippet of text
    - Count the occurrences of each word within it
    - Put the results in a table
  
- We can do this using Scikit-Learn's `CountVectorizer`

In [ ]:
sample = ["to be or not to be", "to kill a mockingbird"]

from sklearn.feature_extraction.text import CountVectorizer

vec = CountVectorizer()
X = vec.fit_transform(sample)
X

- The result is a sparse matrix recording the number of times each word appears
- For inspection a dense `DataFrame` with labeled columns is better

In [ ]:
pd.DataFrame(X.toarray(), columns=vec.get_feature_names_out())

- A simple raw word count can lead to features that put too much weight on words that appear very frequently
- This can be suboptimal in some classification algorithms


- *Term frequency–inverse document frequency* (*TF–IDF*) weights the word counts by a measure of how often they appear in the documents
- *TF–IDF* is often produces more effective models

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer()
X = vec.fit_transform(sample)
pd.DataFrame(X.toarray(), columns=vec.get_feature_names_out())

## Image Features

- Another common need is to encode images for machine learning analysis
- The simplest approach is what we used for the digits data: simply using the pixel values themselves
- Depending on the application, such an approach may not be optimal

- A comprehensive summary of feature extraction techniques for images is well beyond the scope of this section, but you can find excellent implementations of many of the standard approaches in the [Scikit-Image project](http://scikit-image.org)

## Derived Features

- Another useful type of feature is one that is mathematically derived from some input features
- For example, we can construct *polynomial features* from our input data by considering the *polynomial combinations* of the input features


- For an input sample of the form $[a, b]$, this polynomial transformation considers not only $[a, b]$, but also combinations of these features with powers up to a certain degree: $[1, a, b, a^2, ab, b^2]$.

In [ ]:
%matplotlib inline

x = np.array([1, 2, 3, 4, 5])
y = np.array([4, 2, 1, 3, 7])
plt.scatter(x, y);

- Fitting a `LinearRegression` gives a suboptimal fit

In [ ]:
from sklearn.linear_model import LinearRegression

X = x[:, np.newaxis]
model = LinearRegression().fit(X, y)
yfit = model.predict(X)
plt.scatter(x, y)
plt.plot(x, yfit)

- We need a more sophisticated model to describe the relationship between $x$ and $y$


- We can transform the data, adding extra columns of features to drive more flexibility in the model
- For example, we can add polynomial features to the data this way:

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=3, include_bias=False)
X2 = poly.fit_transform(X)
X2

- The derived feature matrix has one column representing $x$, a second column representing $x^2$, and a third column representing $x^3$
- Computing a linear regression on this expanded input gives a much closer fit to our data

In [ ]:
model = LinearRegression().fit(X2, y)
x_new = np.linspace(1, 5, 50)
X_new = poly.transform(x_new.reshape(50, 1))
yfit = model.predict(X_new)
plt.scatter(x, y)
plt.plot(x_new, yfit)

## Imputation of Missing Data

- Another common need is handling missing data
- We've seen that `NaN` is often is used to mark missing values

In [ ]:
from numpy import nan

X = np.array([[nan, 0, 3], [3, 7, 9], [3, 5, 2], [4, nan, 6], [8, 8, 1]])
y = np.array([14, 16, -1, 8, -5])
X

- When applying a typical machine learning model to such data, we first need to replace the missing values
- This is known as *imputation* of missing values
- Imputation strategies range from simple (e.g., replacing missing values with the mean of the column) to sophisticated (e.g., using matrix completion or a robust model to handle such data)


- The sophisticated approaches tend to be very application-specific, and we won't dive into them here
- For a baseline imputation approach using the mean, median, or most frequent value, Scikit-Learn provides the `SimpleImputer` class:

In [ ]:
from sklearn.impute import SimpleImputer

imp = SimpleImputer(strategy="mean")
X2 = imp.fit_transform(X)
X2

- The two missing values have been replaced with the mean of the remaining values in the column
- This imputed data can then be fed directly into, for example, a `LinearRegression` estimator

In [ ]:
# model = LinearRegression().fit(X, y) # throws error
model = LinearRegression().fit(X2, y)
model.predict(X2)

## Feature Pipelines

- It can quickly become tedious to do the transformations by hand, especially if you wish to string together multiple steps
- For example, we might want a processing pipeline that looks something like this:
    1. Impute missing values using the mean
    2. Transform features to quadratic
    3. Fit a linear regression model


- Scikit-Learn provides a ``Pipeline`` object to streamline this process

In [ ]:
from sklearn.pipeline import make_pipeline

model = make_pipeline(SimpleImputer(strategy="mean"), PolynomialFeatures(degree=2), LinearRegression())
model

- This pipeline looks and acts like a standard Scikit-Learn object
- It applies all the specified steps to any input data

In [ ]:
model.fit(X, y)  # X with missing values, from above
print(y)
print(model.predict(X))

# Exercise

- Build a `GaussianNB` classifier to detect the kind of exercise given the diet, pulse, and time (dataset below)
- Most of the work has been done, but the features still need to be extracted
    1. Convert the diet column into features so that they can processed by the model (using sklearn's `OneHotEncoder`)
    2. Extract the numerical value from the time column so that it can be processed by the model (using Pandas)

In [ ]:
exercise = pd.read_csv(
    io.StringIO(requests.get("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/exercise.csv").text),
    usecols=[2, 3, 4, 5]
)
exercise.head()

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

In [ ]:
one_hot_encoder = OneHotEncoder(sparse_output=False)
# TODO build the feature matrix exercise_X and target array exercise_y
# 1. parse the time into an integer
# 2. One-hot encode the kind of exercise

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(exercise_X, exercise_y, random_state=42)

model = GaussianNB()
model.fit(X_train, y_train)
y_model = model.predict(X_test)

accuracy_score(y_test, y_model)